# MicroPhaseLab Lesson 1: From Polygons to Segmentation Masks

This notebook uses synthetic data to explain the first data pipeline. Run `microphaselab demo` first, then execute each cell.

## Learning objectives

1. Distinguish SEM images, polygon annotations, and pixel masks.
2. Understand why masks contain only 0 and 1.
3. Understand why data should be split by sample group rather than random image.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image

root = Path('../examples/demo')
manifest = pd.read_csv(root / 'processed/manifest.csv')
manifest

In [ ]:
row = manifest.iloc[0]
image = np.asarray(Image.open(row.image_path).convert('L'))
mask = np.asarray(Image.open(row.mask_path))
print('image shape:', image.shape)
print('mask values:', np.unique(mask))
print('MA area fraction:', mask.mean())

## Why is pixel accuracy insufficient?

If MA occupies only 5% of an image, an all-background prediction still has 95% accuracy while finding no MA at all. Dice, IoU, precision, and recall are more informative metrics.

## Read the classical baseline results

After running the `microphaselab baseline` command from the README, read the aggregate and per-image metrics. The synthetic bright regions were designed to be easy to segment, so high scores only show that the pipeline works; they do not represent performance on real steel data.

In [ ]:
import json

baseline_root = Path('../outputs/baseline/demo')
summary = json.loads((baseline_root / 'summary.json').read_text(encoding='utf-8'))
metrics = pd.read_csv(baseline_root / 'metrics_per_image.csv')
summary_keys = [
    'mean_dice', 'mean_iou', 'mean_precision', 'mean_recall',
    'mean_area_fraction_absolute_error',
]
{key: summary[key] for key in summary_keys}

### Questions

1. Which image has the lowest Dice score? Inspect its prediction: is the error mainly false positives or false negatives?
2. If area-fraction error is small but IoU is low, what may have happened to the predicted boundary?
3. Why should these two synthetic-image scores not be used to choose parameters for real data?